# Grain-form counts for Table 2 and the density-support figure

Count SnowPilot layers both by exact `Layer.grain_form` code and by two-character parent class. Table 2 uses parent classes; the density-support figure retains its exact method-support rows.

In [1]:
from collections import Counter
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
DATA_DIR = EXAMPLES_DIR / "data"

for path in (PROJECT_ROOT / "src", EXAMPLES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from notebook_utils import load_pits
from paper_figure_utils import SUPPORT_GRAIN_FORM_ORDER as GRAIN_FORM_ORDER

The parser stores a sub-grain code when available and otherwise stores the parent code. Exact-code counts diagnose the original table, while parent-class counts provide a complete, mutually exclusive summary that reconciles to all layers with a recorded primary grain form.

In [2]:
pits = load_pits(str(DATA_DIR))
layers = [layer for pit in pits for layer in pit.layers]
grain_form_counts = Counter(
    layer.grain_form
    for layer in layers
    if layer.grain_form is not None
)
parent_grain_form_counts = Counter(
    grain_form[:2]
    for grain_form, count in grain_form_counts.items()
    for _ in range(count)
)
parent_order = ['FC', 'RG', 'MF', 'DF', 'PP', 'IF', 'DH', 'SH', 'MM']
table_2_summary = pd.DataFrame(
    {
        'grain_form': parent_order,
        'n_layers': [parent_grain_form_counts[code] for code in parent_order],
    }
)
expected_parent_counts = {
    'FC': 79_518, 'RG': 64_884, 'MF': 55_416, 'DF': 43_427,
    'PP': 21_526, 'IF': 19_136, 'DH': 12_379, 'SH': 7_417, 'MM': 23,
}
assert dict(zip(table_2_summary.grain_form, table_2_summary.n_layers)) == expected_parent_counts
assert int(table_2_summary['n_layers'].sum()) == sum(grain_form_counts.values()) == 303_726
assert grain_form_counts['MM'] == 16
assert grain_form_counts['MMrp'] == 6
assert grain_form_counts['MMci'] == 1


summary = pd.DataFrame(
    {
        "grain_form": GRAIN_FORM_ORDER,
        "n_layers": [grain_form_counts.get(grain_form, 0) for grain_form in GRAIN_FORM_ORDER],
    }
)
summary["axis_label"] = summary.apply(
    lambda row: f"{row.grain_form} (n={row.n_layers:,})",
    axis=1,
)

print(f"Loaded {len(pits):,} pits")
print(f"Total layers: {len(layers):,}")
print(f"Layers with grain form: {sum(grain_form_counts.values()):,}")
print('Exact machine-made codes:', {code: grain_form_counts[code] for code in ('MM', 'MMrp', 'MMci')})
display(table_2_summary)
display(summary)

/Users/marykate/Desktop/Snow/SnowPyt-MechParams/.venv/lib/python3.13/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


Loaded 50,278 pits
Total layers: 371,429
Layers with grain form: 303,726
Exact machine-made codes: {'MM': 16, 'MMrp': 6, 'MMci': 1}


,grain_form,n_layers
0,FC,79518
1,RG,64884
2,MF,55416
3,DF,43427
4,PP,21526
5,IF,19136
6,DH,12379
7,SH,7417
8,MM,23


,grain_form,n_layers,axis_label
0,PP,14958,"PP (n=14,958)"
1,PPgp,2582,"PPgp (n=2,582)"
2,MM,16,MM (n=16)
3,DF,29426,"DF (n=29,426)"
4,RG,36017,"RG (n=36,017)"
5,RGxf,6164,"RGxf (n=6,164)"
6,FC,34628,"FC (n=34,628)"
7,FCxr,27445,"FCxr (n=27,445)"
8,DH,7064,"DH (n=7,064)"
9,SH,4612,"SH (n=4,612)"


Use `grain_form_axis_labels` in the support figure if you want the y-axis to include counts.

In [3]:
grain_form_axis_labels = dict(zip(summary["grain_form"], summary["axis_label"]))
grain_form_axis_labels

{'PP': 'PP (n=14,958)',
 'PPgp': 'PPgp (n=2,582)',
 'MM': 'MM (n=16)',
 'DF': 'DF (n=29,426)',
 'RG': 'RG (n=36,017)',
 'RGxf': 'RGxf (n=6,164)',
 'FC': 'FC (n=34,628)',
 'FCxr': 'FCxr (n=27,445)',
 'DH': 'DH (n=7,064)',
 'SH': 'SH (n=4,612)',
 'MF': 'MF (n=6,768)',
 'MFcr': 'MFcr (n=37,997)',
 'IF': 'IF (n=5,577)'}